# Glioblastoma: Segmentação + Graduação (SAM 2 / MedSAM + Atenção)
Projeto final — Visão Computacional Avançada, FCTE/UnB.

Este notebook conecta os módulos `.py` do repositório e demonstra:
1. Setup e sanity-check do pipeline (dataset sintético, sem download)
2. Treino no BraTS real
3. Plots de **loss**, **Dice (WT/TC/ET)** e métricas de **graduação** para o relatório
4. Visualização qualitativa da segmentação


## 1. Setup

In [ ]:
# No Colab, descomente:
# !git clone <SEU_REPO> repo && %cd repo
# !pip install -r requirements.txt
import sys; sys.path.append('.')
import torch
print('CUDA disponível:', torch.cuda.is_available())


## 2. Sanity-check com dataset sintético (valida shapes e loop sem baixar nada)

In [ ]:
from src.train import Trainer, TrainConfig
from src.dataset import SyntheticBraTS
from torch.utils.data import DataLoader

cfg = TrainConfig(backbone='stub', epochs=3, batch_size=8, out_dir='runs/sanity')
tr = DataLoader(SyntheticBraTS(64), batch_size=8, shuffle=True)
va = DataLoader(SyntheticBraTS(16), batch_size=8)
hist = Trainer(cfg).fit(tr, va)


## 2.5 Fine-tuning eficiente com LoRA (opcional, recomendado)

O encoder de fundação fica **congelado**; injetamos adaptadores de baixo posto
(LoRA) só nas projeções de atenção. Treina ~1–5% dos pesos — cabe na T4 do Colab
e preserva os priors do SAM2/MedSAM. `lora_B=0` no início ⇒ parte exatamente do
encoder pré-treinado.


In [ ]:
from src.models import SAM2SegGradeNet
from src.lora import count_trainable

# encoder congelado SEM adaptadores
net_frozen = SAM2SegGradeNet(backbone='stub', use_lora=False)
# encoder congelado COM LoRA (r=8) — só os adaptadores treinam
net_lora   = SAM2SegGradeNet(backbone='stub', use_lora=True, lora_r=8, lora_alpha=16)

s_frozen = count_trainable(net_frozen.encoder)
s_lora   = count_trainable(net_lora.encoder)
print(f"camadas LoRA injetadas       : {net_lora.encoder.n_lora}")
print(f"encoder treinável (congelado): {s_frozen['pct']:.2f}%")
print(f"encoder treinável (LoRA)     : {s_lora['pct']:.2f}%  ({s_lora['trainable']} params)")

# Para treinar com LoRA: TrainConfig(..., use_lora=True) e faça o fit normalmente.
# Para inferência sem custo extra: net_lora.eval(); merge_all_lora(net_lora)


## 3. BraTS real
Monte o Google Drive (ou baixe do TCIA/Synapse) e aponte `root` para a pasta com
subpastas por caso (cada uma com `*_flair.nii.gz`, `*_t1ce.nii.gz`, `*_t2.nii.gz`, `*_seg.nii.gz`).


In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
from src.dataset import make_loaders
from src.train import Trainer, TrainConfig

cfg = TrainConfig(
    root='/content/BraTS2021',   # AJUSTE
    backbone='stub',             # troque p/ 'sam2' com checkpoint (ver README)
    epochs=30, batch_size=8, img_size=256,
    seg_ce_weight=[0.1,1.0,1.0,1.0],   # deprioriza o fundo
    out_dir='runs/brats',
)
# tr, va = make_loaders(cfg.root, batch_size=cfg.batch_size, img_size=cfg.img_size)
# hist = Trainer(cfg).fit(tr, va)


## 4. Curvas para o relatório (loss, Dice por sub-região, graduação)

In [ ]:
from src import viz
import json
with open('runs/sanity/logs/history.json') as f:
    hist = json.load(f)

# Curvas de treino: loss + Dice por sub-região + métricas de graduação
fig = viz.plot_training_curves(hist, save_path='curvas_resultado.png')
fig


## 4.5 Métricas avançadas para o relatório (Hausdorff95 + ROC/AUC)

Além do Dice: **Hausdorff95** mede o erro de borda (mm) por sub-região, e a
**curva ROC / AUC** avalia a graduação LGG×HGG. Aqui usamos o modelo sintético
para demonstrar; no BraTS real, acumule `pred`/`logits` do loader de validação.


In [ ]:
import torch
from src import viz
from src.models import build_model
from src.dataset import SyntheticBraTS
from src.metrics import seg_scores_hd95, grade_roc_auc

model = build_model(cfg).eval()
ds = SyntheticBraTS(64)
imgs   = torch.stack([ds[i]['image']    for i in range(64)])
gts    = torch.stack([ds[i]['seg_mask'] for i in range(64)])
grades = torch.tensor([int(ds[i]['grade']) for i in range(64)])
with torch.no_grad():
    out = model(imgs)
preds        = out['seg_logits'].argmax(1)
grade_logits = out['grade_logits']

# Hausdorff95 por sub-região (nan quando a máscara está vazia — reporte só válidos)
hd = seg_scores_hd95(preds, gts)
print("Hausdorff95 (px):", {k: round(v,2) for k,v in hd.items()})

# ROC / AUC da graduação
roc = grade_roc_auc(grade_logits, grades)
print("AUC graduação   :", round(roc['auc'], 3))

fig_roc  = viz.plot_roc(roc,      save_path='roc_resultado.png')
fig_hd   = viz.plot_hd95_bars(hd, save_path='hd95_resultado.png')
fig_roc


## 5. Visualização qualitativa da segmentação

In [ ]:
import torch
from src import viz
from src.models import build_model
from src.dataset import SyntheticBraTS
from src.metrics import seg_scores

model = build_model(cfg).eval()
ds = SyntheticBraTS(8)
b = ds[0]
with torch.no_grad():
    out = model(b['image'][None])
pred = out['seg_logits'].argmax(1)[0]

# Painel: MRI | Ground Truth | Predição | Mapa de erro (FP magenta / FN ciano)
d = seg_scores(pred[None], b['seg_mask'][None])
fig = viz.qualitative_panel(
    b['image'][0], b['seg_mask'], pred,
    grade_true=int(b['grade']), grade_pred=int(out['grade_logits'].argmax()),
    dice_txt=f"Dice WT={d['dice_WT']:.2f}",
    save_path='qualitativa_resultado.png')
fig
